# Online Distillation: Draft 80M from Qwen2.5-0.5B

**Key difference**: instead of training on static wikitext, the **teacher generates the training data**. The student learns to predict exactly what the teacher would predict.

This directly optimizes for acceptance rate in speculative decoding.

In [ ]:
!pip install -q torch transformers datasets safetensors sentencepiece

In [ ]:
import math, os, time, json, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

device = "cuda"
print(f"GPU: {torch.cuda.get_device_name()}")

## 1. Models

In [ ]:
CFG = {
    "vocab_size": 151936, "hidden_size": 384,
    "num_attention_heads": 6, "num_key_value_heads": 2,
    "intermediate_size": 1536, "num_hidden_layers": 6,
    "rms_norm_eps": 1e-6, "rope_theta": 1000000.0,
    "max_seq_len": 128, "tie_word_embeddings": True,
}

@dataclass
class C:
    vocab_size:int=151936; hidden_size:int=384; num_attention_heads:int=6
    num_key_value_heads:int=2; intermediate_size:int=1536; num_hidden_layers:int=6
    rms_norm_eps:float=1e-6; rope_theta:float=1e6; max_seq_len:int=128
    tie_word_embeddings:bool=True

def rope(hd, sl, th):
    f=1./(th**(torch.arange(0,hd,2).float()/hd))
    e=torch.cat([torch.outer(torch.arange(sl).float(),f)]*2,-1)
    return e.cos(),e.sin()

def rot(x):
    d=x.shape[-1]//2; return torch.cat([-x[...,d:],x[...,:d]],-1)

class RN(nn.Module):
    def __init__(s,d,e=1e-6): super().__init__(); s.w=nn.Parameter(torch.ones(d)); s.e=e
    def forward(s,x): return s.w*(x*torch.rsqrt(x.pow(2).mean(-1,keepdim=True)+s.e))

class At(nn.Module):
    def __init__(s,c):
        super().__init__()
        s.nh,s.nk=c.num_attention_heads,c.num_key_value_heads
        s.hd=c.hidden_size//c.num_attention_heads; s.ng=s.nh//s.nk
        s.q=nn.Linear(c.hidden_size,s.nh*s.hd,bias=True)
        s.k=nn.Linear(c.hidden_size,s.nk*s.hd,bias=True)
        s.v=nn.Linear(c.hidden_size,s.nk*s.hd,bias=True)
        s.o=nn.Linear(s.nh*s.hd,c.hidden_size,bias=False)
    def forward(s,x,co,si,m):
        B,L,_=x.shape
        q=s.q(x).view(B,L,s.nh,s.hd).transpose(1,2)
        k=s.k(x).view(B,L,s.nk,s.hd).transpose(1,2)
        v=s.v(x).view(B,L,s.nk,s.hd).transpose(1,2)
        q=q*co+rot(q)*si; k=k*co+rot(k)*si
        if s.ng>1:
            k=k.unsqueeze(2).expand(B,s.nk,s.ng,L,s.hd).reshape(B,s.nh,L,s.hd)
            v=v.unsqueeze(2).expand(B,s.nk,s.ng,L,s.hd).reshape(B,s.nh,L,s.hd)
        a=F.softmax(torch.matmul(q,k.transpose(-2,-1))/math.sqrt(s.hd)+m,-1)
        return s.o(torch.matmul(a,v).transpose(1,2).contiguous().view(B,L,-1))

class FF(nn.Module):
    def __init__(s,c):
        super().__init__()
        s.g=nn.Linear(c.hidden_size,c.intermediate_size,bias=False)
        s.u=nn.Linear(c.hidden_size,c.intermediate_size,bias=False)
        s.d=nn.Linear(c.intermediate_size,c.hidden_size,bias=False)
    def forward(s,x): return s.d(F.silu(s.g(x))*s.u(x))

class Bk(nn.Module):
    def __init__(s,c):
        super().__init__()
        s.at=At(c);s.ff=FF(c);s.n1=RN(c.hidden_size,c.rms_norm_eps);s.n2=RN(c.hidden_size,c.rms_norm_eps)
    def forward(s,x,co,si,m):
        x=x+s.at(s.n1(x),co,si,m); return x+s.ff(s.n2(x))

class Draft(nn.Module):
    def __init__(s,c):
        super().__init__()
        s.cfg=c; s.emb=nn.Embedding(c.vocab_size,c.hidden_size)
        s.layers=nn.ModuleList([Bk(c) for _ in range(c.num_hidden_layers)])
        s.norm=RN(c.hidden_size,c.rms_norm_eps)
        s.head=nn.Linear(c.vocab_size if False else c.hidden_size,c.vocab_size,bias=False)
        if c.tie_word_embeddings: s.head.weight=s.emb.weight
        hd=c.hidden_size//c.num_attention_heads
        co,si=rope(hd,c.max_seq_len,c.rope_theta)
        s.register_buffer('rc',co.unsqueeze(0).unsqueeze(0))
        s.register_buffer('rs',si.unsqueeze(0).unsqueeze(0))
        mk=torch.triu(torch.full((c.max_seq_len,c.max_seq_len),-1e9),diagonal=1)
        s.register_buffer('mk',mk.unsqueeze(0).unsqueeze(0))
    def forward(s,ids):
        h=s.emb(ids)
        for l in s.layers: h=l(h,s.rc,s.rs,s.mk)
        return s.head(s.norm(h))

student = Draft(C(**CFG)).to(device)
print(f"Student: {sum(p.numel() for p in student.parameters()):,} params")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

teacher = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", torch_dtype=torch.float32).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
print(f"Teacher: {sum(p.numel() for p in teacher.parameters()):,} params")

## 2. Seed prompts from diverse text

In [ ]:
from datasets import load_dataset

# Collect diverse seed prompts (first 10-30 tokens of many texts)
ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
seeds = []
for text in ds["text"]:
    text = text.strip()
    if len(text) > 50:
        ids = tokenizer.encode(text, add_special_tokens=False)
        if len(ids) >= 10:
            # Use first 10-30 tokens as seed
            seed_len = random.randint(10, min(30, len(ids)))
            seeds.append(ids[:seed_len])
    if len(seeds) >= 50000:
        break

print(f"Collected {len(seeds):,} seed prompts")

## 3. Online distillation

Each step:
1. Pick random seed prompts
2. Teacher generates continuation (greedy, 128 tokens)
3. Student trains to match teacher's logits on the FULL sequence

In [ ]:
SEQ = 128
STEPS = 50_000
BATCH = 16
LR = 3e-4
TEMP = 1.5
ALPHA = 0.8  # higher alpha = more focus on matching teacher distribution

@torch.no_grad()
def generate_teacher_batch(batch_size):
    """Teacher generates training sequences from random seeds."""
    # Pick random seeds
    batch_seeds = random.sample(seeds, batch_size)

    # Pad seeds to same length
    max_seed = max(len(s) for s in batch_seeds)
    padded = [s + [tokenizer.eos_token_id] * (max_seed - len(s)) for s in batch_seeds]
    input_ids = torch.tensor(padded, device=device)

    # Generate with teacher (greedy)
    gen_len = SEQ - max_seed
    all_logits = []

    for i in range(gen_len):
        out = teacher(input_ids)
        logits = out.logits[:, -1, :]  # (B, V)
        all_logits.append(logits)
        next_tok = logits.argmax(dim=-1, keepdim=True)  # (B, 1)
        input_ids = torch.cat([input_ids, next_tok], dim=1)

    # Get teacher logits for the FULL sequence (for distillation)
    full_ids = input_ids[:, :SEQ]  # (B, SEQ)
    full_out = teacher(full_ids)
    teacher_logits = full_out.logits  # (B, SEQ, V)

    return full_ids, teacher_logits


# Optimizer
opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)
WU = 1000
def lr_fn(s):
    if s < WU: return s / WU
    return 0.5 * (1 + math.cos(math.pi * (s - WU) / (STEPS - WU)))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

student.train()
losses = []
best = float('inf')
t0 = time.time()

print(f"Online distillation: {STEPS:,} steps | batch={BATCH} | lr={LR} | temp={TEMP} | alpha={ALPHA}")
print("-" * 70)

for step in range(1, STEPS + 1):
    # Generate fresh training data from teacher
    full_ids, t_logits = generate_teacher_batch(BATCH)

    # Student forward on same sequence
    s_logits = student(full_ids)  # (B, SEQ, V_student)

    # Align vocab
    V = min(s_logits.size(-1), t_logits.size(-1))
    sl = s_logits[:, :-1, :V]   # predict next token
    tl = t_logits[:, :-1, :V]
    labels = full_ids[:, 1:]     # target tokens

    # KL divergence (soft targets)
    kl = F.kl_div(
        F.log_softmax(sl / TEMP, dim=-1),
        F.softmax(tl / TEMP, dim=-1),
        reduction="batchmean"
    ) * TEMP**2

    # CE (hard targets)
    ce = F.cross_entropy(sl.reshape(-1, V), labels.reshape(-1))

    loss = ALPHA * kl + (1 - ALPHA) * ce

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    opt.step()
    sched.step()

    losses.append(loss.item())

    if step % 500 == 0:
        avg = sum(losses[-500:]) / 500
        el = time.time() - t0
        eta = el / step * (STEPS - step)
        tag = " *" if avg < best else ""
        if avg < best: best = avg
        print(f"Step {step:6d}/{STEPS:,} | loss={avg:.4f}{tag} | lr={sched.get_last_lr()[0]:.1e} | {el/60:.0f}m | ETA {eta/60:.0f}m")

    if step % 10_000 == 0:
        from safetensors.torch import save_file as sf
        os.makedirs("ckpt", exist_ok=True)
        st = {k: v.cpu() for k, v in student.state_dict().items() if not k.startswith(("rc","rs","mk"))}
        sf(st, f"ckpt/step_{step}.safetensors")
        print(f"  >> saved ckpt/step_{step}.safetensors")

print(f"\nDone! {(time.time()-t0)/60:.0f} min | best={best:.4f}")

## 4. Measure acceptance rate

In [ ]:
student.eval()

prompts = [
    "The meaning of life is",
    "Once upon a time",
    "The capital of France is",
    "In machine learning, a neural network",
    "The president of the United States",
    "Water boils at a temperature of",
    "The largest planet in the solar system",
    "Python is a programming language that",
]

total_match, total_tok = 0, 0

for prompt in prompts:
    ids = tokenizer.encode(prompt)
    s_ctx, t_ctx = list(ids), list(ids)

    for _ in range(50):
        padded = s_ctx[-SEQ:] + [0] * max(0, SEQ - len(s_ctx))
        inp = torch.tensor([padded[:SEQ]], device=device)
        with torch.no_grad(): sl = student(inp)
        pos = min(len(s_ctx), SEQ) - 1
        s_ctx.append(sl[0, pos].argmax().item())

        with torch.no_grad(): tl = teacher(torch.tensor([t_ctx], device=device)).logits
        t_ctx.append(tl[0, -1].argmax().item())

    match = sum(1 for a, b in zip(s_ctx[len(ids):], t_ctx[len(ids):]) if a == b)
    total_match += match; total_tok += 50
    s_txt = tokenizer.decode(s_ctx[len(ids):])
    t_txt = tokenizer.decode(t_ctx[len(ids):])
    print(f'\n"{prompt}"  [{match}/50 = {match/50:.0%}]')
    print(f'  S: {s_txt[:70]}')
    print(f'  T: {t_txt[:70]}')

print(f"\n{'='*60}")
print(f"OVERALL ACCEPTANCE: {total_match}/{total_tok} = {total_match/total_tok:.0%}")
print(f"{'='*60}")
print(f"Target: >30% for useful speculative decoding")

## 5. Save

In [ ]:
from safetensors.torch import save_file
os.makedirs("draft_80m_online", exist_ok=True)
state = {k: v.cpu() for k, v in student.state_dict().items() if not k.startswith(("rc","rs","mk"))}
save_file(state, "draft_80m_online/draft_80m.safetensors")
with open("draft_80m_online/config.json", "w") as f:
    json.dump(CFG, f, indent=2)
print(f"Saved: {os.path.getsize('draft_80m_online/draft_80m.safetensors')/1e6:.0f} MB")

In [ ]:
try:
    from google.colab import files
    files.download("draft_80m_online/draft_80m.safetensors")
    files.download("draft_80m_online/config.json")
except:
    print("Files in draft_80m_online/")